# Lab Assignment 01 — MNIST Classification using ANN/MLP + Hyperparameter Tuning
**Course:** CSET-225 (IMDAI) | **Semester:** 5th (Odd, 2026)

**Contents**
1. Setup & imports
2. Data preparation (normalize, one-hot, split)
3. Model builder — ANN with 3 hidden layers
4. Grid Search
5. Random Search
6. Final training of the best model
7. Evaluation — accuracy, loss, confusion matrix
8. Performance comparison — curves, params, training time
9. Summary & report notes

## 1. Setup & imports

In [ ]:
import os, time, random, itertools, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

## 2. Data Preparation

- Load MNIST from the Keras built-in loader
- Scale pixels to `[0, 1]`
- Flatten `28x28` images to `784`-dim vectors (input to a dense/MLP network)
- One-hot encode the labels
- Carve a validation set out of the training split (54k train / 6k val / 10k test)

In [ ]:
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.mnist.load_data()

print("Raw shapes:", x_train_full.shape, y_train_full.shape, x_test.shape, y_test.shape)
print("Pixel range before scaling:", x_train_full.min(), "-", x_train_full.max())

# --- normalize to [0, 1] ---
x_train_full = x_train_full.astype("float32") / 255.0
x_test_s     = x_test.astype("float32") / 255.0

# --- flatten 28x28 -> 784 ---
x_train_full = x_train_full.reshape(-1, 28 * 28)
x_test_s     = x_test_s.reshape(-1, 28 * 28)

# --- one-hot encode labels ---
NUM_CLASSES = 10
y_train_full_oh = keras.utils.to_categorical(y_train_full, NUM_CLASSES)
y_test_oh       = keras.utils.to_categorical(y_test, NUM_CLASSES)

# --- train / validation split (stratification not needed, MNIST is near-balanced) ---
VAL_SIZE = 6000
idx = np.random.RandomState(SEED).permutation(len(x_train_full))
val_idx, tr_idx = idx[:VAL_SIZE], idx[VAL_SIZE:]

x_train, y_train = x_train_full[tr_idx], y_train_full_oh[tr_idx]
x_val,   y_val   = x_train_full[val_idx], y_train_full_oh[val_idx]

print("train:", x_train.shape, "| val:", x_val.shape, "| test:", x_test_s.shape)
print("Pixel range after scaling:", x_train.min(), "-", x_train.max())
print("Label example (one-hot):", y_train[0])

In [ ]:
# quick look at the data + class balance
fig, axes = plt.subplots(2, 5, figsize=(10, 4.5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i].reshape(28, 28), cmap="gray")
    ax.set_title(f"label: {np.argmax(y_train[i])}")
    ax.axis("off")
plt.suptitle("Sample MNIST training images")
plt.tight_layout(); plt.show()

plt.figure(figsize=(6, 3))
plt.bar(range(10), np.bincount(y_train_full))
plt.title("Class distribution (full training set)")
plt.xlabel("digit"); plt.ylabel("count"); plt.xticks(range(10))
plt.tight_layout(); plt.show()

## 3. Model Implementation — ANN with 3 hidden layers

A single builder function so every search trial creates the same architecture family,
differing only in the hyperparameters we're tuning:

- `units` — width of the three hidden layers
- `activation` — hidden-layer activation
- `dropout` — dropout rate after each hidden layer
- `optimizer_name` + `lr` — optimizer and learning rate

Output layer is always 10 units with softmax; loss is categorical cross-entropy
(matching the one-hot labels).

In [ ]:
def build_mlp(units=(512, 256, 128), activation="relu", dropout=0.2,
              optimizer_name="adam", lr=1e-3, input_dim=784, num_classes=10):
    """Builds a 3-hidden-layer ANN/MLP for MNIST."""
    model = keras.Sequential(name="MLP_3hidden")
    model.add(layers.Input(shape=(input_dim,)))

    for i, u in enumerate(units, start=1):
        model.add(layers.Dense(u, activation=activation, name=f"hidden_{i}"))
        if dropout and dropout > 0:
            model.add(layers.Dropout(dropout, name=f"dropout_{i}"))

    model.add(layers.Dense(num_classes, activation="softmax", name="output"))

    opts = {
        "adam":    keras.optimizers.Adam(learning_rate=lr),
        "sgd":     keras.optimizers.SGD(learning_rate=lr, momentum=0.9),
        "rmsprop": keras.optimizers.RMSprop(learning_rate=lr),
    }
    model.compile(optimizer=opts[optimizer_name.lower()],
                  loss="categorical_crossentropy",
                  metrics=["accuracy"])
    return model


# sanity check
demo = build_mlp()
demo.summary()

## 4. Hyperparameter Tuning — Grid Search

Exhaustive search over a small grid. Each trial trains for a few epochs only
(that's enough to rank configurations) and is scored on **validation accuracy**.

> Tip: on Colab, switch to a GPU runtime (Runtime → Change runtime type → T4 GPU)
> if you widen the grid. On CPU the grid below takes roughly 5–10 minutes.

In [ ]:
def run_trial(params, epochs=6, verbose=0):
    """Trains one config and returns its record + history."""
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)

    model = build_mlp(units=params["units"],
                      activation=params.get("activation", "relu"),
                      dropout=params["dropout"],
                      optimizer_name=params.get("optimizer_name", "adam"),
                      lr=params["lr"])

    t0 = time.time()
    hist = model.fit(x_train, y_train,
                     validation_data=(x_val, y_val),
                     epochs=epochs,
                     batch_size=params["batch_size"],
                     verbose=verbose)
    train_time = time.time() - t0

    val_acc  = float(np.max(hist.history["val_accuracy"]))
    val_loss = float(np.min(hist.history["val_loss"]))

    record = {
        **{k: (str(v) if isinstance(v, tuple) else v) for k, v in params.items()},
        "params_count": int(model.count_params()),
        "val_accuracy": round(val_acc, 4),
        "val_loss": round(val_loss, 4),
        "train_time_s": round(train_time, 1),
    }
    return record, hist, model

In [ ]:
grid = {
    "units":      [(512, 256, 128), (256, 128, 64)],
    "dropout":    [0.2, 0.3],
    "lr":         [1e-3, 5e-4],
    "batch_size": [128],
}

grid_combos = [dict(zip(grid.keys(), combo)) for combo in itertools.product(*grid.values())]
print(f"Grid Search: {len(grid_combos)} configurations\n")

grid_results, grid_histories = [], {}
for i, cfg in enumerate(grid_combos, start=1):
    rec, hist, _ = run_trial(cfg, epochs=6)
    grid_results.append({"search": "grid", "trial": i, **rec})
    grid_histories[f"grid_{i}"] = hist.history
    print(f"[{i}/{len(grid_combos)}] {cfg} -> val_acc={rec['val_accuracy']:.4f} "
          f"({rec['train_time_s']}s)")

grid_df = pd.DataFrame(grid_results).sort_values("val_accuracy", ascending=False)
grid_df

## 5. Hyperparameter Tuning — Random Search

Instead of covering every combination, we sample `N_TRIALS` configurations from a
wider search space. This usually finds a comparable (often better) configuration for
a fraction of the compute, because it explores more distinct values per
hyperparameter.

In [ ]:
N_TRIALS = 8
rng = np.random.RandomState(SEED)

search_space = {
    "units":      [(1024, 512, 256), (512, 256, 128), (256, 128, 64), (128, 128, 64)],
    "dropout":    [0.0, 0.1, 0.2, 0.3, 0.4],
    "lr":         [3e-3, 1e-3, 5e-4, 3e-4, 1e-4],
    "batch_size": [64, 128, 256],
}

def sample_config():
    return {
        "units":      search_space["units"][rng.randint(len(search_space["units"]))],
        "dropout":    float(search_space["dropout"][rng.randint(len(search_space["dropout"]))]),
        "lr":         float(search_space["lr"][rng.randint(len(search_space["lr"]))]),
        "batch_size": int(search_space["batch_size"][rng.randint(len(search_space["batch_size"]))]),
    }

random_results, random_histories = [], {}
for i in range(1, N_TRIALS + 1):
    cfg = sample_config()
    rec, hist, _ = run_trial(cfg, epochs=6)
    random_results.append({"search": "random", "trial": i, **rec})
    random_histories[f"random_{i}"] = hist.history
    print(f"[{i}/{N_TRIALS}] {cfg} -> val_acc={rec['val_accuracy']:.4f} "
          f"({rec['train_time_s']}s)")

random_df = pd.DataFrame(random_results).sort_values("val_accuracy", ascending=False)
random_df

In [ ]:
# combined leaderboard across both searches
all_df = pd.concat([grid_df, random_df], ignore_index=True) \
           .sort_values("val_accuracy", ascending=False) \
           .reset_index(drop=True)

print("Top 5 configurations overall:")
display(all_df.head(5))

best_row = all_df.iloc[0]
best_params = {
    "units":      eval(best_row["units"]) if isinstance(best_row["units"], str) else best_row["units"],
    "dropout":    float(best_row["dropout"]),
    "lr":         float(best_row["lr"]),
    "batch_size": int(best_row["batch_size"]),
}
print("\nBest configuration found:", best_params)
print(f"Found by: {best_row['search']} search | val_accuracy = {best_row['val_accuracy']:.4f}")

## 6. Final Training — best configuration

Retrain the winning configuration for more epochs with `EarlyStopping` (restores the
best weights) so the reported test score comes from a properly converged model.
We also train a plain **baseline** (default hyperparameters, no tuning) so the report
has something to compare the tuned model against.

In [ ]:
EPOCHS = 30

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5,
                                  restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                      patience=3, min_lr=1e-5, verbose=1),
]

tf.keras.backend.clear_session(); tf.random.set_seed(SEED)
best_model = build_mlp(units=best_params["units"], dropout=best_params["dropout"],
                       lr=best_params["lr"])
best_model.summary()

t0 = time.time()
best_history = best_model.fit(x_train, y_train,
                              validation_data=(x_val, y_val),
                              epochs=EPOCHS,
                              batch_size=best_params["batch_size"],
                              callbacks=callbacks,
                              verbose=2)
best_time = time.time() - t0
print(f"\nTuned model training time: {best_time:.1f}s")

In [ ]:
# untuned baseline for comparison
tf.keras.backend.clear_session(); tf.random.set_seed(SEED)
baseline_model = build_mlp(units=(128, 128, 64), dropout=0.0, lr=1e-3)

t0 = time.time()
baseline_history = baseline_model.fit(x_train, y_train,
                                      validation_data=(x_val, y_val),
                                      epochs=EPOCHS,
                                      batch_size=128,
                                      callbacks=callbacks,
                                      verbose=0)
baseline_time = time.time() - t0
print(f"Baseline training time: {baseline_time:.1f}s")

## 7. Model Evaluation — test accuracy, loss, confusion matrix

In [ ]:
base_loss, base_acc = baseline_model.evaluate(x_test_s, y_test_oh, verbose=0)
best_loss, best_acc = best_model.evaluate(x_test_s, y_test_oh, verbose=0)

print(f"Baseline  -> test_loss={base_loss:.4f}  test_accuracy={base_acc:.4f}")
print(f"Tuned MLP -> test_loss={best_loss:.4f}  test_accuracy={best_acc:.4f}")

In [ ]:
y_pred_probs = best_model.predict(x_test_s, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = y_test  # original integer labels

print(classification_report(y_true, y_pred, digits=4))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=range(10)).plot(ax=ax, cmap="Blues",
                                                          colorbar=False, values_format="d")
ax.set_title("Confusion Matrix — Tuned MLP (test set)")
plt.tight_layout(); plt.show()

# normalized version — easier to spot which digits get confused
cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm_norm, display_labels=range(10)).plot(ax=ax, cmap="Blues",
                                                               colorbar=False, values_format=".2f")
ax.set_title("Confusion Matrix (row-normalized)")
plt.tight_layout(); plt.show()

# most frequent confusions
err = cm.copy(); np.fill_diagonal(err, 0)
pairs = [(i, j, err[i, j]) for i in range(10) for j in range(10) if err[i, j] > 0]
pairs.sort(key=lambda t: -t[2])
print("Top misclassifications (true -> predicted : count):")
for t, p, c in pairs[:8]:
    print(f"  {t} -> {p} : {c}")

In [ ]:
# a look at some actual mistakes
wrong = np.where(y_pred != y_true)[0]
print(f"Total misclassified: {len(wrong)} / {len(y_true)}")

fig, axes = plt.subplots(2, 6, figsize=(12, 4.5))
for ax, i in zip(axes.flat, wrong[:12]):
    ax.imshow(x_test[i], cmap="gray")
    ax.set_title(f"true {y_true[i]} / pred {y_pred[i]}", fontsize=9)
    ax.axis("off")
plt.suptitle("Misclassified test samples — Tuned MLP")
plt.tight_layout(); plt.show()

## 8. Performance Comparison — training curves, parameters, training time

In [ ]:
def plot_curves(histories, title):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for name, h in histories.items():
        axes[0].plot(h["accuracy"], label=f"{name} train")
        axes[0].plot(h["val_accuracy"], "--", label=f"{name} val")
        axes[1].plot(h["loss"], label=f"{name} train")
        axes[1].plot(h["val_loss"], "--", label=f"{name} val")
    axes[0].set_title("Accuracy"); axes[0].set_xlabel("epoch"); axes[0].set_ylabel("accuracy")
    axes[1].set_title("Loss");     axes[1].set_xlabel("epoch"); axes[1].set_ylabel("loss")
    for ax in axes:
        ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.suptitle(title)
    plt.tight_layout(); plt.show()

plot_curves({"baseline": baseline_history.history, "tuned": best_history.history},
            "Training vs Validation curves — baseline vs tuned")

In [ ]:
# validation accuracy achieved by every search trial
fig, ax = plt.subplots(figsize=(10, 4))
colors = {"grid": "tab:blue", "random": "tab:orange"}
for s in ["grid", "random"]:
    sub = all_df[all_df["search"] == s]
    ax.scatter(sub["params_count"], sub["val_accuracy"],
               label=f"{s} search", s=60, alpha=0.8, c=colors[s])
ax.set_xlabel("trainable parameters"); ax.set_ylabel("best val accuracy")
ax.set_title("Search trials: model size vs validation accuracy")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
summary = pd.DataFrame([
    {
        "Model": "Baseline MLP (untuned)",
        "Architecture": "784-128-128-64-10",
        "Dropout": 0.0,
        "Learning rate": 1e-3,
        "Batch size": 128,
        "Parameters": baseline_model.count_params(),
        "Test accuracy": round(base_acc, 4),
        "Test loss": round(base_loss, 4),
        "Train time (s)": round(baseline_time, 1),
        "Epochs run": len(baseline_history.history["loss"]),
    },
    {
        "Model": "Tuned MLP (best config)",
        "Architecture": "784-" + "-".join(map(str, best_params["units"])) + "-10",
        "Dropout": best_params["dropout"],
        "Learning rate": best_params["lr"],
        "Batch size": best_params["batch_size"],
        "Parameters": best_model.count_params(),
        "Test accuracy": round(best_acc, 4),
        "Test loss": round(best_loss, 4),
        "Train time (s)": round(best_time, 1),
        "Epochs run": len(best_history.history["loss"]),
    },
])

print("=== FINAL SUMMARY TABLE ===")
display(summary)

total_search_time = all_df["train_time_s"].sum()
print(f"\nGrid search trials  : {len(grid_df)}")
print(f"Random search trials: {len(random_df)}")
print(f"Total tuning time   : {total_search_time:.1f}s")
print(f"Accuracy gain from tuning: {(best_acc - base_acc) * 100:+.2f} percentage points")

In [ ]:
# save artifacts (they land in the Colab file browser; download from the left sidebar)
summary.to_csv("summary_table.csv", index=False)
all_df.to_csv("hyperparameter_search_results.csv", index=False)
best_model.save("mnist_mlp_best.keras")
with open("training_history.json", "w") as f:
    json.dump({"tuned": {k: list(map(float, v)) for k, v in best_history.history.items()},
               "baseline": {k: list(map(float, v)) for k, v in baseline_history.history.items()}}, f)
print("Saved: summary_table.csv, hyperparameter_search_results.csv, mnist_mlp_best.keras, training_history.json")

## 9. Report Notes

Fill these in with the numbers your run produces — the cells above print everything you need.

**Preprocessing.** MNIST loaded via `keras.datasets.mnist`. Pixels scaled from `[0, 255]` to
`[0, 1]` so the inputs sit in a range gradient descent handles well. Each `28x28` image is
flattened to a `784`-dim vector because dense layers take 1-D input. Labels one-hot encoded
into 10-dim vectors to pair with softmax + categorical cross-entropy. Standard split kept
(60k train / 10k test), with 6k held out of train as a validation set — the test set is never
touched during tuning.

**Architecture.** Three hidden dense layers with ReLU, dropout after each for regularization,
and a 10-unit softmax output. Depth was fixed by the assignment; width, dropout, learning rate
and batch size were tuned.

**Training.** Adam optimizer, categorical cross-entropy loss. Search trials ran 6 epochs each
(enough to rank configs); the winning config was retrained for up to 30 epochs with
`EarlyStopping(patience=5, restore_best_weights=True)` and `ReduceLROnPlateau`.

**Tuning.** Grid search covered all combinations of a small grid. Random search sampled 8
configs from a wider space. Discussion point for the report: grid search guarantees coverage
of its grid but cost grows multiplicatively with each hyperparameter, while random search
explores more distinct values per hyperparameter at fixed budget — note which one actually
found your best config, and by how much.

**Evaluation.** Report test accuracy and loss, the per-class precision/recall from the
classification report, and read the confusion matrix: the classic MNIST confusions are 4↔9,
3↔5, 7↔2, which the misclassified-samples grid usually makes obvious (ambiguous handwriting).

**Discussion.** Compare the tuned model to the baseline on accuracy, parameter count, and
training time — a bigger network isn't automatically better, and the scatter plot in section 8
shows whether extra parameters actually bought accuracy. Mention where the MLP plateaus
(~98%) and that the gap to a CNN comes from dense layers discarding spatial structure when
the image is flattened.